In [ ]:
!pip install sentence-transformers -q

In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df_livros = pd.read_csv("/content/drive/MyDrive/datas/books_data.csv")
df_rating = pd.read_csv("/content/drive/MyDrive/datas/Books_rating.csv")

In [ ]:
df_livros["texto_livro"] = (
    df_livros["Title"].fillna("") + " " +
    df_livros["description"].fillna("") + " " +
    df_livros["authors"].fillna("") + " " +
    df_livros["categories"].fillna("")
)

In [ ]:
reviews_agg = (
    df_rating
    .groupby("Title")
    .agg(
        nota_media=("review/score", "mean"),
        qtd_reviews=("review/score", "count"),
        texto_reviews=("review/text", lambda x: " ".join(x.dropna().astype(str).head(10))),
        resumo_reviews=("review/summary", lambda x: " ".join(x.dropna().astype(str).head(10)))
    )
    .reset_index()
)

In [ ]:
base = df_livros.merge(reviews_agg, on="Title", how="left")

base["texto_final"] = (
    base["Title"].fillna("") + " " +
    base["description"].fillna("") + " " +
    base["authors"].fillna("") + " " +
    base["categories"].fillna("") + " " +
    base["resumo_reviews"].fillna("") + " " +
    base["texto_reviews"].fillna("")
)

In [ ]:
modelo = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1")

embeddings_livros = modelo.encode(
    base["texto_final"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3319 [00:00<?, ?it/s]

In [ ]:
def recomendar_por_descricao(descricao, base, embeddings_livros, top_k=10):
    embedding_consulta = modelo.encode(
        [descricao],
        normalize_embeddings=True
    )

    similaridades = cosine_similarity(
        embedding_consulta,
        embeddings_livros
    ).flatten()

    recomendacoes = base.copy()
    recomendacoes["similaridade"] = similaridades

    return (
        recomendacoes
        .sort_values("similaridade", ascending=False)
        [["Title", "authors", "categories", "nota_media", "qtd_reviews", "similaridade"]]
        .head(top_k)
    )

In [ ]:
recomendar_por_descricao(
    "books about food",
    base,
    embeddings_livros,
    top_k=10
)

,Title,authors,categories,nota_media,qtd_reviews,similaridade
17570,Food and Evolution: Toward a Theory of Human F...,"['Marvin Harris', 'Eric B. Ross']",['Cooking'],5.000000,2.0,0.711502
11091,Food in civilization: How history has been aff...,['Carson I. A. Ritchie'],['Civilization'],2.000000,2.0,0.701531
112587,Food and Other Enemies: Stories of Consuming D...,['Leslie Powell'],['Fiction'],5.000000,4.0,0.690471
115345,The Oxford Companion to Food,['Andrew F. Smith'],['Cooking'],4.666667,30.0,0.685944
78199,The getting back to nature diet (A Pivot healt...,NaN,NaN,5.000000,1.0,0.670987
17039,The Encyclopedia of American Food and Drink: W...,['John F. Mariani'],['Cooking'],5.000000,1.0,0.662358
149546,Food: A Culinary History from Antiquity to the...,"['Jean-Louis Flandrin', 'Massimo Montanari']",['Cooking'],4.400000,5.0,0.661261
49467,Christian Science,NaN,NaN,5.000000,1.0,0.659957
91686,Great Adventures in Food: Fresh Ways to Celebr...,"['Ann Cooper', 'Lisa M. Holmes']",['History'],5.000000,2.0,0.656673
73197,The Garden of Eating,['Jeremy Iggers'],['Psychology'],5.000000,1.0,0.652103


In [ ]:
base.to_csv("base_livros_recomendador.csv", index=False)
np.save("embeddings_livros.npy", embeddings_livros)

In [ ]:
colunas_app = [
    "Title",
    "authors",
    "categories",
    "nota_media",
    "qtd_reviews",
    "ratingsCount"
]

base_app = base[colunas_app].copy()

base_app["nota_media"] = base_app["nota_media"].fillna(0)
base_app["qtd_reviews"] = base_app["qtd_reviews"].fillna(0)
base_app["ratingsCount"] = base_app["ratingsCount"].fillna(0)

base_app.to_parquet("base_livros_app.parquet", index=False)

embeddings_reduzidos = embeddings_livros.astype("float16")
np.save("embeddings_livros_float16.npy", embeddings_reduzidos)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

descricao = "fantasy books with elves, dragons, magic kingdoms and epic adventures"

embedding_consulta = modelo.encode(
    [descricao],
    normalize_embeddings=True
)

# Embeddings originais
sim_float32 = cosine_similarity(
    embedding_consulta,
    embeddings_livros
).flatten()

# Embeddings convertidos
embeddings_float16 = embeddings_livros.astype("float16").astype("float32")

sim_float16 = cosine_similarity(
    embedding_consulta,
    embeddings_float16
).flatten()

top10_float32 = np.argsort(sim_float32)[::-1][:10]
top10_float16 = np.argsort(sim_float16)[::-1][:10]

print("Top 10 float32:", top10_float32)
print("Top 10 float16:", top10_float16)

intersecao = len(set(top10_float32).intersection(set(top10_float16)))

print(f"Livros em comum no Top 10: {intersecao}/10")

Top 10 float32: [ 25400 155610 211687 205042  20415 148888  37710  58958 118653 115804]
Top 10 float16: [ 25400 155610 211687 205042  20415 148888  37710  58958 118653 115804]
Livros em comum no Top 10: 10/10
